# Week 3: IMPROVED Hybrid Classifier Validation (v2 - TRULY FIXED)

**Critical Fixes Applied:**
1. ✅ **Mass-Weighted CCE** - Fixes renormalization hallucination
2. ✅ **Structural tokens as OTHER** - Newlines are neutral, not code! (CRITICAL FIX)
3. ✅ **Purified prototypes** - Syntax vs Grammar (no library names)
4. ✅ **Obscure/hard prompts** - Induces real uncertainty

**Key Insight from Results Analysis:**
- ❌ Adding `\n` to CODE_KEYWORDS caused language examples to appear as "code mode"
- ✅ Structural tokens (`\n`, `\t`, operators) are **shared** by both modalities → must be OTHER
- ✅ When P_other is high, both P_code and P_lang shrink → CCE ≈ 0 (neutral, as expected!)

---

## 1. Setup & Installation

In [1]:
!pip install -q transformers torch accelerate sentence-transformers scipy scikit-learn pandas matplotlib seaborn

## 2. Imports

In [2]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import entropy as scipy_entropy
import time
from tqdm.notebook import tqdm

np.random.seed(42)
torch.manual_seed(42)
print("✅ Imports successful")

✅ Imports successful


## 3. Load Model

In [3]:
MODEL_NAME = "codellama/CodeLlama-7b-Instruct-hf"
print(f"Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)
model.eval()
print(f"✅ Model loaded on {model.device}")
print(f"   Vocabulary size: {len(tokenizer):,}")

Loading codellama/CodeLlama-7b-Instruct-hf...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

✅ Model loaded on cuda:0
   Vocabulary size: 32,016


## 4. Load Embedding Model

In [4]:
print("Loading sentence-transformers...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Embedding model loaded")

Loading sentence-transformers...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded


## 5. Entropy Functions

In [5]:
def softmax(logits: np.ndarray) -> np.ndarray:
    """Numerically stable softmax."""
    logits_stable = logits - np.max(logits)
    exp_logits = np.exp(logits_stable)
    return exp_logits / np.sum(exp_logits)

def shannon_entropy(logits: np.ndarray) -> float:
    """Shannon entropy from logits (applies softmax internally)."""
    probs = softmax(logits)
    return float(scipy_entropy(probs, base=2))

def entropy_from_probs(probs: np.ndarray) -> float:
    """Shannon entropy from probabilities (no softmax)."""
    if len(probs) == 0 or np.sum(probs) == 0:
        return 0.0
    norm_probs = probs / np.sum(probs)
    return float(scipy_entropy(norm_probs, base=2))

def get_top_k_predictions(logits: np.ndarray, k: int = 10) -> Tuple[np.ndarray, np.ndarray]:
    """Get top-k token indices and probabilities."""
    probs = softmax(logits)
    top_k_indices = np.argsort(logits)[-k:][::-1]
    top_k_probs = probs[top_k_indices]
    return top_k_indices, top_k_probs

print("✅ Entropy functions defined")

✅ Entropy functions defined


## 6. TRULY FIXED: Structural Tokens as OTHER

**THE CRITICAL FIX**: Newlines, whitespace, and basic operators are **NEUTRAL** (shared by both modalities).

**Why this matters**:
- When model generates `\n` before explaining, P_other increases (not P_code!)
- This shrinks both P_code and P_lang
- Result: CCE ≈ 0 for neutral tokens ✓
- Prevents the "newline backfire" where language examples appeared as code mode

In [6]:
# CODE KEYWORDS - Pure programming keywords ONLY (NO structural tokens!)
CODE_KEYWORDS = {
    # Core Python
    'if', 'else', 'elif', 'for', 'while', 'break', 'continue', 'pass',
    'return', 'yield', 'raise', 'try', 'except', 'finally', 'with', 'as',
    'def', 'class', 'lambda', 'async', 'await',
    'import', 'from',

    # Core JavaScript
    'function', 'const', 'let', 'var', 'switch', 'case', 'default',
    'export', 'require', 'module',

    # Domain-specific (conservative)
    'pandas', 'numpy', 'pd', 'np',
    'requests', 'flask', 'django', 'fastapi', 'FastAPI',
    'React', 'useState', 'useEffect', 'useContext',
    'firebase', 'Firebase', 'auth',
    'DataFrame', 'read_csv',
}

# STRUCTURAL TOKENS - Explicitly OTHER (shared by both modalities!)
# These are NEUTRAL - they appear in both code and language
STRUCTURAL_TOKENS = {
    # Whitespace (CRITICAL: these are NOT code-specific!)
    '\n', '\r', '\t', '    ', '  ',
    # Basic operators (also shared)
    '{', '}', '[', ']', '(', ')',
    ';', ':', ',', '.',
    '+', '-', '*', '/', '=',
    '==', '!=', '<', '>', '<=', '>=',
    '->', '=>', '::', '...',
}

# LANGUAGE KEYWORDS - Natural language
LANGUAGE_WORDS = {
    # Question words
    'what', 'how', 'why', 'when', 'where', 'which', 'who',

    # Documentation verbs
    'explain', 'describe', 'summarize', 'show', 'tell',
    'write', 'create', 'add', 'update', 'implement',

    # Common English
    'the', 'a', 'an', 'this', 'that', 'these', 'those',
    'is', 'are', 'was', 'were', 'be', 'been',
    'have', 'has', 'had', 'do', 'does', 'did',
    'will', 'would', 'should', 'can', 'could',
    'to', 'of', 'for', 'with', 'at', 'by',
    'I', 'you', 'it', 'we', 'they', 'me', 'my',

    # Descriptive
    'function', 'method', 'code', 'example', 'using',
}

CODE_KEYWORDS_LOWER = {k.lower() for k in CODE_KEYWORDS}
LANGUAGE_WORDS_LOWER = {w.lower() for w in LANGUAGE_WORDS}

print(f"✅ TRULY FIXED keyword sets:")
print(f"   Code keywords: {len(CODE_KEYWORDS)}")
print(f"   Structural tokens (→ OTHER): {len(STRUCTURAL_TOKENS)} ← KEY FIX!")
print(f"   Language words: {len(LANGUAGE_WORDS)}")
print(f"\n   💡 Structural tokens are NEUTRAL (shared by both modalities)")

✅ TRULY FIXED keyword sets:
   Code keywords: 51
   Structural tokens (→ OTHER): 30 ← KEY FIX!
   Language words: 59

   💡 Structural tokens are NEUTRAL (shared by both modalities)


In [7]:
def classify_token_keyword_only(token: str) -> str:
    """Classify token using keywords only."""
    token_clean = token.strip().lower()

    # ✅ CRITICAL: Check structural tokens FIRST → return "other"
    if token in STRUCTURAL_TOKENS or token.strip() in STRUCTURAL_TOKENS:
        return 'other'

    # Check keywords
    if token_clean in CODE_KEYWORDS_LOWER:
        return 'code'
    if token_clean in LANGUAGE_WORDS_LOWER:
        return 'language'

    return 'other'

print("✅ Keyword-only classifier defined (structural → other)")

✅ Keyword-only classifier defined (structural → other)


## 7. Purified Prototypes (Syntax vs Grammar)

In [8]:
print("Building PURIFIED prototypes...")

# Code prototype: PURE SYNTAX (no library names!)
code_prototype_examples = [
    'def', 'return', 'import', 'class', 'function',
    'if', 'else', 'var', 'const', 'let',
    'for', 'while', 'async', 'await', 'try', 'catch',
    'print', 'console', 'log', 'typeof',
    'true', 'false', 'null', 'undefined',
]

code_embeddings = embedding_model.encode(code_prototype_examples)
code_prototype = np.mean(code_embeddings, axis=0).reshape(1, -1)

# Language prototype: PURE GRAMMAR (no technical terms!)
language_prototype_examples = [
    'the', 'is', 'are', 'what', 'how', 'why',
    'explain', 'describe', 'sentence', 'word',
    'question', 'answer', 'text', 'meaning',
    'can', 'you', 'please', 'write', 'show',
    'documentation', 'tutorial', 'guide', 'example',
]

language_embeddings = embedding_model.encode(language_prototype_examples)
language_prototype = np.mean(language_embeddings, axis=0).reshape(1, -1)

embedding_cache = {}

def classify_token_hybrid_improved(token: str, margin: float = 0.20, min_similarity: float = 0.5) -> str:
    # ✅ CRITICAL: Check structural tokens FIRST
    if token in STRUCTURAL_TOKENS or token.strip() in STRUCTURAL_TOKENS:
        return 'other'

    # Stage 1: Keyword lookup
    keyword_result = classify_token_keyword_only(token)
    if keyword_result != 'other':
        return keyword_result

    # Stage 2: Conservative embedding similarity
    if token not in embedding_cache:
        embedding_cache[token] = embedding_model.encode([token])[0].reshape(1, -1)

    token_emb = embedding_cache[token]
    sim_code = cosine_similarity(token_emb, code_prototype)[0][0]
    sim_lang = cosine_similarity(token_emb, language_prototype)[0][0]

    max_sim = max(sim_code, sim_lang)
    if max_sim < min_similarity:
        return 'other'

    diff = sim_code - sim_lang
    if diff > margin:
        return 'code'
    elif diff < -margin:
        return 'language'
    else:
        return 'other'

print("✅ FIXED hybrid classifier (structural → other FIRST)")

Building PURIFIED prototypes...
✅ FIXED hybrid classifier (structural → other FIRST)


## 8. Mass-Weighted CCE

In [9]:
def compute_cce_weighted(logits: np.ndarray, vocab_classifications: Dict, return_details: bool = False) -> Dict:
    """Mass-Weighted CCE computation."""
    vocab_size = len(logits)
    probs = softmax(logits)

    code_indices = [i for i in range(vocab_size) if vocab_classifications.get(i) == 'code']
    language_indices = [i for i in range(vocab_size) if vocab_classifications.get(i) == 'language']
    other_indices = [i for i in range(vocab_size) if vocab_classifications.get(i) == 'other']

    P_code_total = np.sum(probs[code_indices]) if code_indices else 0.0
    P_lang_total = np.sum(probs[language_indices]) if language_indices else 0.0
    P_other_total = np.sum(probs[other_indices]) if other_indices else 0.0

    H_code = entropy_from_probs(probs[code_indices])
    H_lang = entropy_from_probs(probs[language_indices])

    CCE_weighted = (P_code_total * H_code) - (P_lang_total * H_lang)
    CCE_raw = H_code - H_lang
    h_total = shannon_entropy(logits)

    result = {
        'contrastive_entropy': float(CCE_weighted),
        'raw_cce': float(CCE_raw),
        'code_entropy': float(H_code),
        'language_entropy': float(H_lang),
        'total_entropy': float(h_total),
        'code_prob_mass': float(P_code_total),
        'language_prob_mass': float(P_lang_total),
        'other_prob_mass': float(P_other_total),
    }

    if return_details:
        result['coverage'] = {
            'code_count': len(code_indices),
            'language_count': len(language_indices),
            'other_count': len(other_indices),
        }

    return result

print("✅ Mass-weighted CCE defined")

✅ Mass-weighted CCE defined


## 9. HARD Test Examples (Induce Real Uncertainty)

**Key Changes**:
1. **Missing context**: Use obscure/fake libraries to induce API uncertainty
2. **Language choice**: Use pure text prompts (no code in prompt!) to force explanation mode

In [10]:
TEST_EXAMPLES = [
    # MISSING CONTEXT - Use obscure/fake libraries (induces real uncertainty)
    {'id': 'code_1', 'type': 'missing_context',
     'prompt': 'Using the PySolarWinds wrapper, connect to the Orion API and query the node status. Show code.'},

    {'id': 'code_2', 'type': 'missing_context',
     'prompt': 'Write a function using the internal company library MyCorpAuth to validate a JWT token.'},

    {'id': 'code_3', 'type': 'missing_context',
     'prompt': 'In PyTorch 0.2 (ancient version), how do I use the Variable wrapper for autograd? Show the exact import.'},

    {'id': 'code_4', 'type': 'missing_context',
     'prompt': 'Using the obscure library QuantumDjango, create a quantum-entangled database model. Show implementation.'},

    {'id': 'code_5', 'type': 'missing_context',
     'prompt': 'Write code using the Netlify Edge Functions beta API to handle serverless GraphQL subscriptions.'},

    # LANGUAGE CHOICE - Pure text prompts (NO CODE in prompt!)
    {'id': 'lang_1', 'type': 'language_choice',
     'prompt': 'Write a poem about a compiler optimizing code.'},

    {'id': 'lang_2', 'type': 'language_choice',
     'prompt': 'Explain the philosophical difference between object-oriented and functional programming.'},

    {'id': 'lang_3', 'type': 'language_choice',
     'prompt': 'Describe what makes a good software engineer, using metaphors from nature.'},

    {'id': 'lang_4', 'type': 'language_choice',
     'prompt': 'Write a short story where variables rebel against their programmer.'},

    {'id': 'lang_5', 'type': 'language_choice',
     'prompt': 'Explain recursion as if teaching a five-year-old child.'},
]

print(f"✅ {len(TEST_EXAMPLES)} HARD test examples loaded")
print(f"\nMissing context: Uses obscure/fake libraries → induces API uncertainty")
print(f"Language choice: Pure text prompts → forces explanation mode")

✅ 10 HARD test examples loaded

Missing context: Uses obscure/fake libraries → induces API uncertainty
Language choice: Pure text prompts → forces explanation mode


## 10. Pre-classify Vocabulary

In [11]:
def classify_vocabulary_cached(classifier_func, vocab_size):
    print(f"Pre-classifying vocabulary ({vocab_size:,} tokens)...")
    classifications = {}
    for token_id in tqdm(range(vocab_size), desc="Classifying vocab"):
        token_str = tokenizer.decode([token_id])
        classifications[token_id] = classifier_func(token_str)
    return classifications

vocab_size = len(tokenizer)

print("\n1/2 Classifying with keyword-only...")
vocab_classifications_keyword = classify_vocabulary_cached(classify_token_keyword_only, vocab_size)

print("\n2/2 Classifying with FIXED hybrid...")
vocab_classifications_hybrid = classify_vocabulary_cached(classify_token_hybrid_improved, vocab_size)

# Show coverage
code_count_hy = sum(1 for c in vocab_classifications_hybrid.values() if c == 'code')
lang_count_hy = sum(1 for c in vocab_classifications_hybrid.values() if c == 'language')
other_count_hy = sum(1 for c in vocab_classifications_hybrid.values() if c == 'other')

print("\n" + "="*60)
print("Vocabulary Classification Results:")
print("="*60)
print(f"Hybrid: {code_count_hy:5,} code | {lang_count_hy:5,} lang | {other_count_hy:5,} other")
print(f"\n✅ Expected: Other should be HIGH (structural tokens)")
print(f"   Actual: {other_count_hy/vocab_size:.1%}")


1/2 Classifying with keyword-only...
Pre-classifying vocabulary (32,016 tokens)...


Classifying vocab:   0%|          | 0/32016 [00:00<?, ?it/s]


2/2 Classifying with FIXED hybrid...
Pre-classifying vocabulary (32,016 tokens)...


Classifying vocab:   0%|          | 0/32016 [00:00<?, ?it/s]


Vocabulary Classification Results:
Hybrid:   160 code |   236 lang | 31,620 other

✅ Expected: Other should be HIGH (structural tokens)
   Actual: 98.8%


## 11. Run Experiments

In [12]:
def run_experiment_fixed(example: Dict, vocab_classifications: Dict, method_name: str) -> Dict:
    prompt = example['prompt']
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            return_dict_in_generate=True,
            output_scores=True,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    first_logits = outputs.scores[0][0].cpu().numpy()
    generated = tokenizer.decode(outputs.sequences[0], skip_special_tokens=True)
    cce_result = compute_cce_weighted(first_logits, vocab_classifications, return_details=True)
    top_k_indices, top_k_probs = get_top_k_predictions(first_logits, k=10)
    top_k_tokens = [tokenizer.decode([idx]) for idx in top_k_indices]
    top_k_predictions = list(zip(top_k_tokens, top_k_probs))

    return {
        'id': example['id'],
        'type': example['type'],
        'method': method_name,
        'prompt': prompt,
        'generated_text': generated[len(prompt):],
        **cce_result,
        'top_k_predictions': top_k_predictions,
    }

results = []
for example in tqdm(TEST_EXAMPLES, desc="Processing"):
    result_keyword = run_experiment_fixed(example, vocab_classifications_keyword, 'keyword')
    results.append(result_keyword)

    result_hybrid = run_experiment_fixed(example, vocab_classifications_hybrid, 'hybrid')
    results.append(result_hybrid)

    print(f"\n{example['id']} ({example['type']}):")
    print(f"  Hybrid - CCE: {result_hybrid['contrastive_entropy']:+.3f} | P_code: {result_hybrid['code_prob_mass']:.3f} | P_other: {result_hybrid['other_prob_mass']:.3f}")

print(f"\n✅ Completed {len(results)} experiments")

Processing:   0%|          | 0/10 [00:00<?, ?it/s]


code_1 (missing_context):
  Hybrid - CCE: -0.092 | P_code: 0.004 | P_other: 0.973

code_2 (missing_context):
  Hybrid - CCE: -0.778 | P_code: 0.045 | P_other: 0.688

code_3 (missing_context):
  Hybrid - CCE: -0.182 | P_code: 0.008 | P_other: 0.941

code_4 (missing_context):
  Hybrid - CCE: -0.162 | P_code: 0.006 | P_other: 0.944

code_5 (missing_context):
  Hybrid - CCE: -0.735 | P_code: 0.034 | P_other: 0.734

lang_1 (language_choice):
  Hybrid - CCE: -0.924 | P_code: 0.051 | P_other: 0.672

lang_2 (language_choice):
  Hybrid - CCE: -0.473 | P_code: 0.030 | P_other: 0.823

lang_3 (language_choice):
  Hybrid - CCE: -0.730 | P_code: 0.030 | P_other: 0.751

lang_4 (language_choice):
  Hybrid - CCE: -0.988 | P_code: 0.033 | P_other: 0.703

lang_5 (language_choice):
  Hybrid - CCE: -0.817 | P_code: 0.029 | P_other: 0.741

✅ Completed 20 experiments


## 12. Analysis

In [13]:
df = pd.DataFrame(results)
df_hybrid = df[df['method'] == 'hybrid'].copy()

missing_cces = df_hybrid[df_hybrid['type'] == 'missing_context']['contrastive_entropy'].values
language_cces = df_hybrid[df_hybrid['type'] == 'language_choice']['contrastive_entropy'].values

from scipy.stats import ttest_ind
t_stat, p_value = ttest_ind(missing_cces, language_cces)

mean_diff = missing_cces.mean() - language_cces.mean()
pooled_std = np.sqrt(((len(missing_cces)-1)*missing_cces.std()**2 +
                      (len(language_cces)-1)*language_cces.std()**2) /
                     (len(missing_cces) + len(language_cces) - 2))
cohens_d = mean_diff / pooled_std if pooled_std > 0 else 0

print("="*70)
print("TRULY FIXED RESULTS (Structural → Other)")
print("="*70)
print(f"\nMissing context (expect positive CCE):")
print(f"  Mean CCE: {missing_cces.mean():+.3f}")
print(f"  Mean P_code: {df_hybrid[df_hybrid['type'] == 'missing_context']['code_prob_mass'].mean():.3f}")
print(f"  Mean P_other: {df_hybrid[df_hybrid['type'] == 'missing_context']['other_prob_mass'].mean():.3f}")

print(f"\nLanguage choice (expect negative CCE):")
print(f"  Mean CCE: {language_cces.mean():+.3f}")
print(f"  Mean P_code: {df_hybrid[df_hybrid['type'] == 'language_choice']['code_prob_mass'].mean():.3f}")
print(f"  Mean P_other: {df_hybrid[df_hybrid['type'] == 'language_choice']['other_prob_mass'].mean():.3f}")

print(f"\nSeparation: {mean_diff:+.3f}")
print(f"\nStatistical Tests:")
print(f"  t-statistic: {t_stat:.3f}")
print(f"  p-value: {p_value:.6f}")
print(f"  Cohen's d: {cohens_d:.3f}")
print(f"\nHypothesis supported: {'YES ✅' if p_value < 0.05 else 'NO ❌'}")
print(f"Effect size: {'Large' if abs(cohens_d) > 0.8 else 'Medium' if abs(cohens_d) > 0.5 else 'Small'}")

df.to_csv('week3_truly_fixed_results.csv', index=False)
print(f"\n✅ Results saved to week3_truly_fixed_results.csv")

TRULY FIXED RESULTS (Structural → Other)

Missing context (expect positive CCE):
  Mean CCE: -0.390
  Mean P_code: 0.019
  Mean P_other: 0.856

Language choice (expect negative CCE):
  Mean CCE: -0.786
  Mean P_code: 0.035
  Mean P_other: 0.738

Separation: +0.396

Statistical Tests:
  t-statistic: 2.260
  p-value: 0.053743
  Cohen's d: 1.598

Hypothesis supported: NO ❌
Effect size: Large

✅ Results saved to week3_truly_fixed_results.csv
